# Observed Station Precipitation Aggregation
## Reference Period: 1995–2014

**Purpose:**  
Read daily observed precipitation records for the 49 selected stations across Jordan's hydrological basins and aggregate them to:

- **Monthly totals** (mm/month) — one value per station per calendar month per year  
- **Long-term monthly means** (mm/month) — climatological average across 1995–2014  
  (12 values per station, used as the observational benchmark in model evaluation)

**Inputs:**  
- `Station_Basin_Mapping.xlsx` — 49 stations with IDs, names, basin assignments  
- `daily.for.stations/Station_<ID>_Daily_Rainfall.xlsx` — raw daily records  

**Outputs saved to:** `C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data\`

**Data sources:**  
Ministry of Water and Irrigation, Jordan | Ministry of Agriculture, Jordan  
All station records underwent quality control prior to delivery.



$$\text{MS}(y, m) = \sum_{d=1}^{n} P(y, m, d)$$

$$\text{LTMA}(m) = \frac{1}{Y} \sum_{y=1}^{Y} \text{MS}(y, m)$$

where $P$ is daily precipitation (mm/day), $y$ is year, $m$ is month, and $Y = 20$ years.


## 1. Import Libraries


In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

print("Libraries loaded.")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")


Libraries loaded.
  numpy  : 2.1.3
  pandas : 2.2.3


## 2. Configuration

Update the paths below if your directory structure differs.


In [2]:
# ── Input paths ──────────────────────────────────────────────────────────────
STATION_MAPPING_FILE = (
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

DAILY_DATA_DIR = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS\daily.for.stations"
)

# ── Output root ───────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data"
)

MONTHLY_OUT = OUTPUT_ROOT / "monthly"
LTMM_OUT    = OUTPUT_ROOT / "long_term_monthly_mean"

for d in [MONTHLY_OUT, LTMM_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# ── Period ────────────────────────────────────────────────────────────────────
PERIOD_START = 1995
PERIOD_END   = 2014

MONTH_NAMES = {
    1: "January",  2: "February", 3: "March",    4: "April",
    5: "May",      6: "June",     7: "July",      8: "August",
    9: "September",10: "October", 11: "November", 12: "December"
}

print("Configuration loaded.")
print(f"  Period        : {PERIOD_START}–{PERIOD_END}")
print(f"  Daily data    : {DAILY_DATA_DIR}")
print(f"  Output root   : {OUTPUT_ROOT}")


Configuration loaded.
  Period        : 1995–2014
  Daily data    : D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS\daily.for.stations
  Output root   : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data


## 3. Load Station Mapping

The 49 selected stations and their basin assignments are read from `Station_Basin_Mapping.xlsx`.


In [3]:
stations = pd.read_excel(STATION_MAPPING_FILE)
stations["Station_ID"]   = stations["Station_ID"].astype(str).str.strip()
stations["Station_Name"] = stations["Station_Name"].astype(str).str.strip()
stations["Basin"]        = stations["Basin"].astype(str).str.strip()

print(f"Stations loaded  : {len(stations)}")
print(f"Unique basins    : {stations['Basin'].nunique()}")
print()
print(stations[["Station_ID","Station_Name","Basin"]].to_string(index=False))


Stations loaded  : 49
Unique basins    : 12

Station_ID                        Station_Name                 Basin
    AB0004                      KH.EL-WAHADNEH               N.R.S.W
    AD0002                              HARTHA      YARMOUK (JORDAN)
    AD0005                             UM QEIS      YARMOUK (JORDAN)
    AD0008                              KHARJA      YARMOUK (JORDAN)
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN)
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN)
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN)
    AE0003                           KAFR YUBA               N.R.S.W
    AE0004                           KAFR ASAD               N.R.S.W
    AH0002                       WADI EL-YABIS JORDAN VALLY (JORDAN)
    AJ0001                   AJLUN POLICE POST               N.R.S.W
    AL0008                                ALUK  AMMAN ZARQA (JORDAN)
    AL0010              DEIR ALLA AGR. STATION JORDAN VALL

## 4. Station File Matching

Daily Excel files are named `Station_<ID>_Daily_Rainfall.xlsx` (e.g. `Station_AB0004_Daily_Rainfall.xlsx`, `Station_F 0002_Daily_Rainfall.xlsx`). A normalisation function handles case, whitespace, and underscore differences between the station ID in the mapping file and the filename.


In [4]:
def norm(s: str) -> str:
    """Lowercase and collapse whitespace/underscores/hyphens to a single space."""
    return re.sub(r"[\s_\-]+", " ", str(s).strip().lower())


def find_station_file(station_id: str, all_files: list) -> str | None:
    """
    Match a station_id string to its daily rainfall xlsx file.
    Extracts the token between 'Station_' and '_Daily' in the filename,
    normalises both strings, then compares.
    """
    sid_norm = norm(station_id)
    for f in all_files:
        base = os.path.splitext(f)[0]
        m = re.search(r"^[Ss]tation[_ ](.+?)[_ ][Dd]aily", base)
        file_id = norm(m.group(1)) if m else norm(base)
        if file_id == sid_norm or sid_norm in norm(base):
            return f
    return None


# Scan available xlsx files once
all_xlsx = [f for f in os.listdir(DAILY_DATA_DIR) if f.lower().endswith(".xlsx")]
print(f"Excel files found in daily data directory: {len(all_xlsx)}")

# Pre-match all stations so we can report coverage up front
match_report = []
for _, stn in stations.iterrows():
    fname = find_station_file(stn["Station_ID"], all_xlsx)
    match_report.append({
        "Station_ID"  : stn["Station_ID"],
        "Station_Name": stn["Station_Name"],
        "Basin"       : stn["Basin"],
        "File"        : fname if fname else "NOT FOUND",
        "Status"      : "OK" if fname else "MISSING",
    })

match_df = pd.DataFrame(match_report)
n_found   = (match_df["Status"] == "OK").sum()
n_missing = (match_df["Status"] == "MISSING").sum()

print(f"Matched  : {n_found} / {len(stations)}")
if n_missing:
    print(f"Missing  : {n_missing}")
    print(match_df[match_df["Status"] == "MISSING"][["Station_ID","Station_Name"]].to_string(index=False))


Excel files found in daily data directory: 62
Matched  : 49 / 49


## 5. Aggregate Daily Observations to Monthly Totals

For each station:
1. Read the daily Excel file  
2. Filter rows to the evaluation period 1995–2014  
3. Sum daily values within each calendar month → monthly total (mm/month)  
4. Record the number of valid days per month for data-quality awareness

All stations are saved in a **single combined CSV**: `monthly/obs_monthly_all_stations.csv`  
Columns: `Station_ID`, `Station_Name`, `Basin`, `Year`, `Month`, `pr_mm_month`, `N_Valid_Days`

> **Missing-data convention:** If a station-month has **zero** data rows in the source file, the monthly total is recorded as `NaN` (not zero) to distinguish true missing months from genuinely dry months.


In [5]:
monthly_out_csv = MONTHLY_OUT / "obs_monthly_all_stations.csv"

all_monthly_rows = []
log_rows = []

for _, stn in tqdm(stations.iterrows(), total=len(stations), desc="Stations"):
    sid   = stn["Station_ID"]
    sname = stn["Station_Name"]
    basin = stn["Basin"]

    fname = find_station_file(sid, all_xlsx)

    if fname is None:
        log_rows.append({"Station_ID": sid, "Station_Name": sname,
                         "Status": "FILE NOT FOUND", "N_Months": 0})
        continue

    fpath = DAILY_DATA_DIR / fname
    try:
        df = pd.read_excel(fpath, parse_dates=["Date"])
    except Exception as e:
        log_rows.append({"Station_ID": sid, "Station_Name": sname,
                         "Status": f"READ ERROR: {e}", "N_Months": 0})
        continue

    # Clean
    df["Date"]     = pd.to_datetime(df["Date"], errors="coerce")
    df["Rainfall"] = pd.to_numeric(df["Rainfall"], errors="coerce")
    df = df.dropna(subset=["Date"])
    df["Year"]  = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month

    # Filter period
    df = df[df["Year"].between(PERIOD_START, PERIOD_END)].copy()

    if df.empty:
        log_rows.append({"Station_ID": sid, "Station_Name": sname,
                         "Status": "NO DATA IN PERIOD", "N_Months": 0})
        continue

    # Monthly aggregation — count valid (non-NaN) days separately from sum
    grp = df.groupby(["Year","Month"])
    monthly_sum  = grp["Rainfall"].sum()    # NaN days treated as 0 by sum
    monthly_nval = grp["Rainfall"].count()  # number of non-NaN rows

    n_months = 0
    for yr in range(PERIOD_START, PERIOD_END + 1):
        for mo in range(1, 13):
            key = (yr, mo)
            if key in monthly_sum.index:
                n_valid = int(monthly_nval[key])
                # If ALL days in this month were NaN → treat as missing
                pr_val  = monthly_sum[key] if n_valid > 0 else np.nan
                all_monthly_rows.append({
                    "Station_ID"   : sid,
                    "Station_Name" : sname,
                    "Basin"        : basin,
                    "Year"         : yr,
                    "Month"        : mo,
                    "pr_mm_month"  : round(float(pr_val), 4) if not np.isnan(pr_val) else np.nan,
                    "N_Valid_Days" : n_valid,
                })
                n_months += 1
            else:
                # Month entirely absent from file → NaN
                all_monthly_rows.append({
                    "Station_ID"   : sid,
                    "Station_Name" : sname,
                    "Basin"        : basin,
                    "Year"         : yr,
                    "Month"        : mo,
                    "pr_mm_month"  : np.nan,
                    "N_Valid_Days" : 0,
                })

    log_rows.append({"Station_ID": sid, "Station_Name": sname,
                     "Status": "OK", "N_Months": n_months})

monthly_df = pd.DataFrame(all_monthly_rows)
monthly_df.sort_values(["Station_ID","Year","Month"], inplace=True)
monthly_df.to_csv(monthly_out_csv, index=False)

log_df = pd.DataFrame(log_rows)
print(f"Monthly aggregation complete.")
print(f"  Total station-month rows : {len(monthly_df):,}")
print(f"  Output CSV               : {monthly_out_csv}")
print()
print("Processing log:")
print(log_df.to_string(index=False))


Stations:   0%|          | 0/49 [00:00<?, ?it/s]

Monthly aggregation complete.
  Total station-month rows : 11,760
  Output CSV               : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data\monthly\obs_monthly_all_stations.csv

Processing log:
Station_ID                        Station_Name Status  N_Months
    AB0004                      KH.EL-WAHADNEH     OK       240
    AD0002                              HARTHA     OK       240
    AD0005                             UM QEIS     OK       240
    AD0008                              KHARJA     OK       240
    AD0011                         EN NUEIYIME     OK       240
    AD0023                     JABER MUGHAYYIR     OK       240
    AD0032     BAQURA MET.STATION (METEO DEPT)     OK       240
    AE0003                           KAFR YUBA     OK       240
    AE0004                           KAFR ASAD     OK       240
    AH0002                       WADI EL-YABIS     OK       240
    AJ0001                   AJLUN POLICE POST     OK       240
    AL0008      

## 6. Compute Long-Term Monthly Means (Observed Climatology)

The long-term monthly mean (LTMM) is the average of the monthly totals across all years in the evaluation period for each month of the year. This gives 12 values per station representing the observed climatological seasonal cycle.

Only months with valid (non-NaN) data contribute to the mean. The `N_Years` column records how many years had valid data for each month — useful for identifying data gaps that may affect evaluation metric reliability.

Saved to: `long_term_monthly_mean/obs_ltmm_all_stations.csv`  
Columns: `Station_ID`, `Station_Name`, `Basin`, `Month`, `Month_Name`, `LTMM_mm_month`, `N_Years`


In [6]:
ltmm_out_csv = LTMM_OUT / "obs_ltmm_all_stations.csv"

ltmm_df = (
    monthly_df
    .groupby(["Station_ID","Station_Name","Basin","Month"], sort=True)["pr_mm_month"]
    .agg(LTMM_mm_month="mean", N_Years="count")   # count excludes NaN automatically
    .reset_index()
)
ltmm_df["LTMM_mm_month"] = ltmm_df["LTMM_mm_month"].round(4)
ltmm_df["Month_Name"]    = ltmm_df["Month"].map(MONTH_NAMES)

# Reorder columns
ltmm_df = ltmm_df[["Station_ID","Station_Name","Basin",
                     "Month","Month_Name","LTMM_mm_month","N_Years"]]
ltmm_df.to_csv(ltmm_out_csv, index=False)

print(f"LTMM computation complete.")
print(f"  Rows saved : {len(ltmm_df):,}  ({len(stations)} stations × 12 months)")
print(f"  Output CSV : {ltmm_out_csv}")
print()
print("Preview — first station (12 months):")
first_stn = ltmm_df["Station_ID"].iloc[0]
print(ltmm_df[ltmm_df["Station_ID"] == first_stn]
      [["Station_ID","Station_Name","Basin","Month_Name",
        "LTMM_mm_month","N_Years"]].to_string(index=False))


LTMM computation complete.
  Rows saved : 588  (49 stations × 12 months)
  Output CSV : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv

Preview — first station (12 months):
Station_ID   Station_Name   Basin Month_Name  LTMM_mm_month  N_Years
    AB0004 KH.EL-WAHADNEH N.R.S.W    January         87.490       20
    AB0004 KH.EL-WAHADNEH N.R.S.W   February         81.740       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      March         53.025       20
    AB0004 KH.EL-WAHADNEH N.R.S.W      April         11.580       20
    AB0004 KH.EL-WAHADNEH N.R.S.W        May          7.360       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       June          0.000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W       July          0.000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W     August          0.000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W  September          0.000       20
    AB0004 KH.EL-WAHADNEH N.R.S.W    October         11.700       20

## 7. Data Coverage Summary

A summary table showing, for each station, how many months have valid observed data within 1995–2014 (maximum possible = 240 months = 20 years × 12 months).  
Low coverage stations should be noted when interpreting validation metrics.


In [7]:
coverage = (
    monthly_df
    .groupby(["Station_ID","Station_Name","Basin"])["pr_mm_month"]
    .agg(
        N_Months_Total   = "count",      # non-NaN months
        N_Months_Possible= lambda x: len(x),  # total rows (including NaN)
    )
    .reset_index()
)
coverage["Coverage_pct"] = (
    coverage["N_Months_Total"] / coverage["N_Months_Possible"] * 100
).round(1)
coverage["Flag"] = coverage["Coverage_pct"].apply(
    lambda p: "⚠ LOW (<75%)" if p < 75 else "✓ OK"
)

coverage_csv = OUTPUT_ROOT / "station_data_coverage.csv"
coverage.to_csv(coverage_csv, index=False)

print("Data coverage summary:")
print(coverage[["Station_ID","Station_Name","Basin",
                "N_Months_Total","Coverage_pct","Flag"]].to_string(index=False))
print()
print(f"Coverage table saved: {coverage_csv}")


Data coverage summary:
Station_ID                        Station_Name                 Basin  N_Months_Total  Coverage_pct Flag
    AB0004                      KH.EL-WAHADNEH               N.R.S.W             240         100.0 ✓ OK
    AD0002                              HARTHA      YARMOUK (JORDAN)             240         100.0 ✓ OK
    AD0005                             UM QEIS      YARMOUK (JORDAN)             240         100.0 ✓ OK
    AD0008                              KHARJA      YARMOUK (JORDAN)             240         100.0 ✓ OK
    AD0011                         EN NUEIYIME      YARMOUK (JORDAN)             240         100.0 ✓ OK
    AD0023                     JABER MUGHAYYIR      YARMOUK (JORDAN)             240         100.0 ✓ OK
    AD0032     BAQURA MET.STATION (METEO DEPT) JORDAN VALLY (JORDAN)             240         100.0 ✓ OK
    AE0003                           KAFR YUBA               N.R.S.W             240         100.0 ✓ OK
    AE0004                           KAFR

## 8. Output Directory Summary


In [8]:
print("=" * 65)
print("OUTPUT DIRECTORY STRUCTURE")
print("=" * 65)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    level   = Path(root).relative_to(OUTPUT_ROOT).parts
    indent  = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub = "  " * (len(level) + 1)
    for f in sorted(files):
        sz = (Path(root)/f).stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub}📄 {f}  ({sz:.1f} {unit})")

print()
print("Next step → run Notebook 03_Validation.ipynb")


OUTPUT DIRECTORY STRUCTURE
📁 stations data/
  📄 station_data_coverage.csv  (2.7 KB)
  📁 long_term_monthly_mean/
    📄 obs_ltmm_all_stations.csv  (29.6 KB)
  📁 monthly/
    📄 obs_monthly_all_stations.csv  (554.0 KB)

Next step → run Notebook 03_Validation.ipynb
